In [ ]:
# final string worked before mixed last run data is 16-6-2026

# -*- coding: utf-8 -*-
"""AcceptedTerm-organized.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1m6HuAnzBe_2IcILero_NHxIVkfhz1dKW
"""

# project/
# │
# ├── main.py
# ├── config.py
# ├── loader.py
# ├── processor.py
# ├── rules.py
# ├── models.py
# ├── writer.py
# └── utils.py

# config.py
CONFIG = {
    "value_column": "ValueName",
    "id_column": "ValueID",
    "accepted_terms_column": "Accepted Terms",
    "output_file": "result.xlsx"
}

# loader.py
import pandas as pd

# def load_data(input_file, acc_terms_file, acc_column):
#     df = pd.read_excel(input_file)
#     acc_df = pd.read_excel(acc_terms_file)

#     acc_terms_set = set(
#         acc_df[acc_column]
#         .dropna()
#         .astype(str)
#         .str.strip()

#     )

#     return df, acc_terms_set
_cache = {}

def load_data(input_file, acc_terms_file, acc_column):

    global _cache

    if acc_terms_file in _cache:
        acc_terms_set = _cache[acc_terms_file]
    else:
        acc_df = pd.read_excel(acc_terms_file)

        acc_terms_set = set(
            acc_df[acc_column]
            .fillna("")
            .astype(str)
            .str.strip()

        )
        acc_terms_set_Lower = set(
            acc_df[acc_column]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
        )

        _cache[acc_terms_file] = acc_terms_set  # 🔥 CACHE IT

    df = pd.read_excel(input_file)

    return df, acc_terms_set, acc_terms_set_Lower

# # utils.py
# import re

# def split_groups(value: str):
#     return [g.strip() for g in re.split(r", \s*", value) if g.strip()]

# def split_values(group: str):
#     return [v.strip() for v in re.split(r", \s*", group) if v.strip()]
# def extract_value_parts(val: str):
#     val = val.strip()

#     # Extract number
#     num_match = re.search(r"-?\d+\.?\d*", val)
#     number = num_match.group() if num_match else ""

#     # Extract unit (everything except number)
#     unit = re.sub(r"-?\d+\.?\d*", "", val).strip()

#     return number, unit

# def protect_identifiers(text):
#     """
#     Protect text inside parentheses if parentheses are NOT
#     at the beginning of the string.
#     """

#     placeholders = {}
#     counter = 0

#     pattern = r'(?<!^)\([^()]*\)'

#     def replacer(match):
#         nonlocal counter

#         original = match.group(0)
#         key = f"__ID_{counter}__"

#         placeholders[key] = original
#         counter += 1

#         return key

#     protected_text = re.sub(pattern, replacer, text)

#     return protected_text, placeholders


# def restore_identifiers(text, placeholders):
#     """
#     Restore protected identifiers.
#     """

#     for key, value in placeholders.items():
#         text = text.replace(key, value)

#     return text

import re


# ============================================================
# IDENTIFIER PROTECTION
# ============================================================



def preprocess_identifier(value: str):

    """
    Extract identifiers inside parentheses
    while preserving exact format spacing.

    Returns:
        actual_string
        identifier
        format_string
    """

    if not value:
        return "", "", ""

    original = str(value)

    identifiers = []

    format_parts = []

    actual_parts = []

    pattern = r"\([^()]*\)"

    last_end = 0

    for match in re.finditer(pattern, original):

        start, end = match.span()

        # -----------------------------------
        # TEXT BEFORE IDENTIFIER
        # -----------------------------------
        text_part = original[last_end:start]

        if text_part:

            actual_parts.append(text_part)

            # Preserve spacing logic
            stripped = text_part.strip()

            if stripped:

                leading_spaces = len(text_part) - len(text_part.lstrip(" "))
                trailing_spaces = len(text_part) - len(text_part.rstrip(" "))

                if leading_spaces:
                    format_parts.append(" " * leading_spaces)

                format_parts.append("T")

                if trailing_spaces:
                    format_parts.append(" " * trailing_spaces)

        # -----------------------------------
        # IDENTIFIER
        # -----------------------------------
        identifier_text = match.group(0)[1:-1]

        identifiers.append(identifier_text)

        leading_spaces = len(identifier_text) - len(identifier_text.lstrip(" "))
        trailing_spaces = len(identifier_text) - len(identifier_text.rstrip(" "))
        format_parts.append(
            "(" + (" " * leading_spaces) + "I" + (" " * trailing_spaces) + ")"
        )

        last_end = end

    # -----------------------------------
    # REMAINING TEXT
    # -----------------------------------
    remaining = original[last_end:]

    if remaining:

        actual_parts.append(remaining)

        stripped = remaining.strip()

        if stripped:

            leading_spaces = len(remaining) - len(remaining.lstrip(" "))
            trailing_spaces = len(remaining) - len(remaining.rstrip(" "))

            if leading_spaces:
                format_parts.append(" " * leading_spaces)

            format_parts.append("T")

            if trailing_spaces:
                format_parts.append(" " * trailing_spaces)

    # -----------------------------------
    # CLEAN ACTUAL STRING
    # -----------------------------------
    actual_string = "".join(actual_parts)

    actual_string = re.sub(pattern, "", actual_string)

    actual_string = re.sub(r"\s+", " ", actual_string).strip()

    # -----------------------------------
    # FINAL FORMAT
    # -----------------------------------
    format_string = "".join(format_parts)

    format_string = format_string.strip()

    identifier_string = ", ".join(
        x.strip() for x in identifiers if x.strip()
    )

    return actual_string, identifier_string, format_string
def protect_identifiers(text):

    placeholders = {}
    counter = 0

    # Protect any (...) NOT at beginning
    pattern = r'(?<!^)\([^()]*\)'

    def replacer(match):

        nonlocal counter

        original = match.group(0)

        key = f"__ID_{counter}__"

        placeholders[key] = original

        counter += 1

        return key

    protected_text = re.sub(pattern, replacer, text)

    return protected_text, placeholders


def restore_identifiers(text, placeholders):

    for key, value in placeholders.items():
        text = text.replace(key, value)

    return text


# ============================================================
# FLAGS
# ============================================================

def has_identifier(val: str):

    """
    Identifier = parentheses exist
    BUT value does NOT start with '('
    """

    if not val:
        return "No"

    val = val.strip()

    if val.startswith("("):
        return "No"

    return "Yes" if re.search(r"\([^()]*\)", val) else "No"


def starts_with_parenthesis(val: str):

    """
    YES only if FIRST character is '('
    """

    if not val:
        return "No"

    return "Yes" if val.lstrip().startswith("(") else "No"
def have_one_parenthesis(val: str):
    if not val:
        return "No"

    s = val.lstrip()
    return "Yes" if "(" in s and ")" not in s else "No"
# ============================================================
# SPLIT GROUPS
# ============================================================

def split_groups(value: str):

    if not value:
        return []

    protected_value, placeholders = protect_identifiers(value)

    groups = [
        restore_identifiers(g.strip(), placeholders)
        for g in re.split(r";\s*", protected_value)
        if g.strip()
    ]

    return groups


# ============================================================
# SPLIT VALUES
# ============================================================

def split_values(group: str):

    if not group:
        return []

    protected_group, placeholders = protect_identifiers(group)

    values = [
        restore_identifiers(v.strip(), placeholders)
        for v in re.split(r",\s*", protected_group)
        if v.strip()
    ]

    return values


# ============================================================
# VALUE PARTS
# ============================================================

def extract_value_parts(val: str):

    val = val.strip()

    num_match = re.search(r"-?\d+\.?\d*", val)

    number = num_match.group() if num_match else ""

    unit = re.sub(r"-?\d+\.?\d*", "", val).strip()

    return number, unit

def remove_after_x(text: str):
    if not text:
        return text
    return re.split(r"\s+x", text)[0].strip()
def has_trailing_space(value: str):
    """
    Detect normal trailing spaces.
    """
    if not isinstance(value, str):
        return "No"

    return "Yes" if value.endswith(" ") else "No"


def has_nbsp(value: str):
    """
    Detect non-breaking spaces (\u00A0).
    """
    if not isinstance(value, str):
        return "No"

    return "Yes" if "\u00A0"  in value else "No"

def has_capitalization_difference(value: str,acc_terms_set,acc_terms_set_lower):

    """
    Detect capitalization-only differences.
    """

    if not isinstance(value, str):
        return "No"

    value_original = value.strip()

    value_clean = value_original.lower()

    # exists logically
    if value_clean in acc_terms_set_lower:

        # exact match exists
        if value_original in acc_terms_set:
            return "No"

        return "Yes"

    return "No"

# # rules.py
# class RuleEngine:

#     def __init__(self, acc_terms_set):
#         self.acc_terms_set = acc_terms_set
#         self.acc_terms_set_Lower = acc_terms_set

#     def classify(self, val: str):
#         # val_clean = val.lower()

#         if val in self.acc_terms_set:
#             return "Accepted", "^"

#         # 🔥 Add future rules here
#         # if "ohm" in val_clean:
#         #     return "Electrical", "#"

#         return "Free Text", "@"
class RuleEngine:

    def __init__(self, acc_terms_set):

        # original
        self.acc_terms_set = set(
            x.strip()
            for x in acc_terms_set
        )

        # lowercase cache
        self.acc_terms_set_lower = set(
            x.lower()
            for x in self.acc_terms_set
        )

    def classify(self, val: str):

        val_original = val.strip()

        val_clean = val_original.lower()

        # ✅ accepted term
        if val_clean in self.acc_terms_set_lower:
            return "Accepted", "^"

        return "Free Text", "@"

# models.py
# class ParsedValue:
#     def __init__(self, value_id, original, parsed, value_type, pattern,identifier="No",starts_with_parenthesis="No"):
#         self.value_id = value_id
#         self.original = original
#         self.parsed = parsed
#         self.type = value_type
#         self.pattern = pattern

#         self.identifier = identifier
#         self.starts_with_parenthesis = starts_with_parenthesis

class ParsedValue:

    def __init__(self,value_id,original,actual_string,identifier,format_string,parsed,value_type,pattern,starts_with_parenthesis="No",has_trailing_space="No",has_nbsp="No",capitalization_difference="No",have_one_parenthesis_flag="No"):
        self.value_id = value_id
        self.original = original

        self.actual_string = actual_string
        self.identifier = identifier
        self.format = format_string

        self.parsed = parsed
        self.type = value_type
        self.pattern = pattern

        self.starts_with_parenthesis = starts_with_parenthesis
        # ✅ NEW FLAGS
        self.has_trailing_space = has_trailing_space
        self.has_nbsp = has_nbsp
        self.capitalization_difference = capitalization_difference
        self.have_one_parenthesis_flag = have_one_parenthesis_flag


class PatternResult:
    def __init__(self, value_id, original, pattern, detailed_type,detailed_type_new, value_type):
        # self.value_id = value_id
        # self.original = original
        # self.pattern = pattern
        # self.detailed_type = detailed_type
        # self.value_type = value_type
        #
        self.value_id = value_id
        self.original = original
        self.pattern = pattern
        self.detailed_type = detailed_type
        self.detailed_type_new = detailed_type_new
        self.value_type = value_type



class ControllerResult:
    def __init__(self, value_id, detailed_type, attr_name, feature_code,detailed_type_new, attr_name_new, feature_code_new):
        self.value_id = value_id
        self.detailed_type = detailed_type
        self.attr_name = attr_name
        self.feature_code = feature_code
        self.detailed_type_new = detailed_type_new
        self.attr_name_new__s = attr_name_new
        self.feature_code_new = feature_code_new

# from utils import split_groups, split_values
# from models import ParsedValue, PatternResult, ControllerResult
# from rules import RuleEngine


class ValueProcessor:

    def __init__(self, acc_terms_set, config):
        self.rule_engine = RuleEngine(acc_terms_set)
        # self.rule_engine = RuleEngine(acc_terms_set)

        self.config = config

    # ============================================================
    # MAIN PIPELINE
    # ============================================================
    def process_row(self, row):

        value_col = self.config["value_column"]
        id_col = self.config["id_column"]

        # original_value = str(row[value_col]) if row[value_col] else ""
        original_value = str(row[value_col]) if row[value_col] else ""

        # ============================================================
        # PREPROCESS IDENTIFIER
        # ============================================================

        actual_string, identifier_value, format_string = preprocess_identifier(original_value)
        ## add these when work to add Identifier row
        has_identifier_value = (isinstance(identifier_value, str)and identifier_value.strip() != "")
        ##
        value_id = row[id_col]
        data_def = row.get("DataDefinition", "")
        accepted_feature = row.get("AcceptedFeatureName", "")
        part_count = row.get("CountRows", "")

        expanded = []
        controller = []

        # groups = split_groups(original_value)
        groups = split_groups(actual_string)

        full_symbols = []
        total_accepted = 0
        total_free = 0
        free_positions = []
        temp_values = []
        position_counter = 1

        # ============================================================
        # FLATTEN ALL VALUES (GLOBAL LOGIC)
        # ============================================================
        value_type = "Free Text"   # ← ADD THIS LINE

        for group in groups:
            values = split_values(group)

            # for val in values:
            #     value_type, symbol = self.rule_engine.classify(val)
            for val in values:

                identifier_flag = has_identifier(val)

                starts_with_flag = starts_with_parenthesis(val)
                have_one_parenthesis_flag = have_one_parenthesis(val)
                # ✅ NEW FLAGS
                trailing_space_flag = has_trailing_space(original_value)

                nbsp_flag = has_nbsp(original_value)
                # ✅ ADD THIS
                capitalization_flag = has_capitalization_difference(
                    val,
                    self.rule_engine.acc_terms_set,
                    self.rule_engine.acc_terms_set_lower
                )

                value_type, symbol = self.rule_engine.classify(val)
                if value_type == "Accepted":
                    total_accepted += 1
                else:
                    total_free += 1
                    free_positions.append(position_counter)

                full_symbols.append(symbol)

                # expanded.append(
                #     ParsedValue(value_id, original_value, val, value_type)
                # )
                # temp_values.append((val, value_type))
                # temp_values.append((val,value_type,identifier_flag,starts_with_flag))
                # These
                temp_values.append((val,value_type,identifier_flag,starts_with_flag,trailing_space_flag,nbsp_flag,actual_string,identifier_value,format_string,capitalization_flag,have_one_parenthesis_flag))
                # ============================================
                # ADD IDENTIFIER AS SECOND VALUE
                # ============================================

                if has_identifier_value:
                    print("Skip")
                    # temp_values.append((
                    #     identifier_value.strip(),
                    #     "Identifier",
                    #     "Yes",
                    #     "No",
                    #     "No",
                    #     "No",
                    #     actual_string,
                    #     identifier_value,
                    #     format_string,
                    #     "No",
                    #     have_one_parenthesis_flag
                    # ))
                # position_counter += 1

        total = total_accepted + total_free
        # ✅ ADD HERE
        feature_prefix = "M0" if total == 1 else "M1"
        # ============================================================
        # PATTERN STRING
        # ============================================================
        pattern_string = ", ".join(full_symbols)
        #
        # for val, value_type in temp_values:
        #   expanded.append(
        #       # ParsedValue(
        #       #     value_id,
        #       #     original_value,
        #       #     val,
        #       #     value_type,
        #       #     pattern_string,   # ✅ NEW

        #       #     )
        #       ParsedValue(
        #           value_id,
        #           original_value,
        #           val,
        #           value_type,
        #           pattern_string,
        #           identifier_flag,
        #           starts_with_flag
        #       )
        #   )
        # for val, value_type, identifier_flag, starts_with_flag in temp_values:

        for (val,value_type,identifier_flag,starts_with_flag,trailing_space_flag,nbsp_flag,actual_string,identifier_value,format_string,capitalization_flag,have_one_parenthesis_flag) in temp_values:
          expanded.append(
              ParsedValue(
                  value_id,
                  original_value,
                  actual_string,
                  identifier_value,
                  format_string,
                  val,
                  value_type,
                  pattern_string,
                  starts_with_flag,
                  trailing_space_flag,
                  nbsp_flag,
                  capitalization_flag,
                  have_one_parenthesis_flag
              )
          )
        # ============================================================
        # DETAILED VALUE TYPE (FIXED LOGIC)
        # ============================================================
        if total == 1 and has_identifier_value:
            if total_accepted == total:
                detailed_type = f"Single String [1][0] x{total}"
            else:
                detailed_type = f"Single String [0][1] x{total}"
            detailed_type_new = f"Single String with Identifier [1][0] x{total}"
        elif total_accepted == total and total !=1:
            detailed_type = f"Multiple (Single String) [1][0] x{total}"
            detailed_type_new = f"Multiple (Single String) [1][0] x{total}"
        elif total_accepted == total and total ==1:
            detailed_type = f"Single String [1][0] x{total}"
            detailed_type_new = f"Single String [1][0] x{total}"
        elif total_free == total and total ==1:
            detailed_type = f"Single String [0][1] x{total}"
            detailed_type_new = f"Single String [1][0] x{total}"
        elif total_free == total:
            detailed_type = f"Multiple (Single String) [0][1] x{total}"
            detailed_type_new = f"Multiple (Single String) [1][0] x{total}"

        else:
            free_pos = ",".join(map(str, free_positions))
            detailed_type = (
                f"Mixed (AcceptedTerm, free text {free_pos}) "
                f"[{total_accepted}][{total_free}] x{total}"
            )
            detailed_type_new = (
                f"Mixed (AcceptedTerm, free text {free_pos}) "
                f"[{total_accepted}][{total_free}] x{total}"
            )

        # ============================================================
        # CONTROLLER BUILDING
        # ============================================================

        controller = self.build_controller(
            value_id,
            total_accepted,
            total_free,
            total,
            detailed_type,
            detailed_type_new,

        )
        pattern = PatternResult(
            value_id,
            original_value,
            pattern_string,
            detailed_type,
            detailed_type_new,
            value_type
        )
        pattern_str = pattern.pattern
        detailed_type = pattern.detailed_type
        detailed_type_new = pattern.detailed_type_new
        # Map parsed values by order
        parsed_values = [x.parsed for x in expanded]
        # ============================================================
        # COMBINED TABLE
        # ============================================================
        all_combined = []
        detailed_type_new = pattern.detailed_type_new

        pattern_str = pattern.pattern
        detailed_type = pattern.detailed_type
        detailed_type_new = pattern.detailed_type_new



        combined_values = [
            item for item in temp_values
            if item[1] != "Identifier"
        ]

        # for idx, (val, value_type) in enumerate(combined_values, start=1):
        # for idx, (val,value_type,identifier_flag,starts_with_flag) in enumerate(combined_values, start=1):
        for idx, (val,value_type,identifier_flag,starts_with_flag,trailing_space_flag,nbsp_flag,actual_string,identifier_value,format_string,capitalization_flag,have_one_parenthesis_flag) in enumerate(combined_values, start=1):
            # ✅ Dynamic prefix per position
            if total == 1:
                feature_prefix = "M0"
                attr_name_new = f"Main string - P0/M0"
                feature_code_new = f"{feature_prefix}-ST-P0"
            else:
                feature_prefix = f"M{idx}"
                attr_name_new = f"Main string - P{idx}/M0"
                feature_code_new = f"{feature_prefix}-ST-P{idx}"

            if value_type == "Accepted":
                attr_name = f"Accepted-Term P{idx}"
                feature_code = f"{feature_prefix}-AT-P{idx}"
                # attr_name_new = f"String P{idx}"
                # feature_code_new = f"{feature_prefix}-ST-P{idx}"
            else:
                attr_name = f"FreeText P{idx}"
                feature_code = f"{feature_prefix}-FT-P{idx}"
                # attr_name_new = f"String P{idx}"
                # feature_code_new = f"{feature_prefix}-ST-P{idx}"

            # ✅ NEW

            try:
                num, unit = extract_value_parts(val)
            except:
                num, unit = "", ""

            attribute_value = original_value if total == 1 and has_identifier_value else val
            combined_identifier_flag = "Yes" if has_identifier_value else identifier_flag

            combined_row = {
                "ValueID": value_id,
                "AcceptedValue": original_value,
                "AttributeValue": attribute_value,
                "AttributeName": attr_name,
                # "FeatureCode": feature_code,
                # "DetailedValueType": detailed_type,
                "Pattern": pattern_string,
                "AttributeName_new": attr_name_new,
                "FeatureCode_new": feature_code_new,
                "DetailedValueTypeNew": detailed_type_new,
                "AbsolutePattern": pattern_string,
                # ✅ NEW FIELDS
                "DataDefinition": data_def,
                "AcceptedFeatureName": accepted_feature,
                "PartCount": part_count,

                "AbsolutePattern": pattern_string,
                "Identifier": combined_identifier_flag,
                "StartsWithParenthesis": starts_with_flag
                # "NormalizedValue": num,
                # "Unit_Identifier": unit,
                # "Unit": unit,
                # "Identifier": "",
            }

            all_combined.append(combined_row)

        return expanded, pattern, controller, all_combined

    # ============================================================
    # CONTROLLER
    # ============================================================

    def build_controller(self, value_id, accepted_count, free_count, total, detailed_type,detailed_type_new):

        results = []

        is_single = total == 1

        idx = 1

        # Accepted Terms
        for i in range(1, accepted_count + 1):

            prefix = "M0" if is_single else f"M{idx}"

            results.append(
                ControllerResult(
                    value_id,
                    detailed_type,
                    f"Accepted-Term P{i}",
                    f"{prefix}-AT-P{i}",
                    detailed_type_new,
                    "Main string - M0" if is_single else f"Main string - P{i}/M{idx}",
                    f"{prefix}-ST-P0" if is_single else f"{prefix}-ST-P{i}"
                )
            )

            idx += 0 if is_single else 1

        # Free Text
        for i in range(1, free_count + 1):

            prefix = "M0" if is_single else f"M{idx}"

            results.append(
                ControllerResult(
                    value_id,
                    detailed_type,
                    f"FreeText P{i}",
                    f"{prefix}-FT-P{i}",
                    detailed_type_new,
                    "Main string - M0" if is_single else f"Main string - P{i}/M{idx}",
                    f"{prefix}-ST-P0" if is_single else f"{prefix}-ST-P{i}"


                )
            )

            idx += 0 if is_single else 1

        return results

# writer.py
import pandas as pd
from openpyxl.styles import PatternFill, Font, Alignment

# =========================
# TARGET COLUMNS STYLE
# =========================
target_fill = PatternFill(
    start_color="FFF2CC",   # Light yellow
    end_color="FFF2CC",
    fill_type="solid"
)

target_font = Font(
    bold=True
)

# =========================
# APPLY ONLY TO:
# FeatureCode
# AttributeName
# =========================
target_columns = ["FeatureCode", "AttributeName"]
# def save_output(expanded, patterns, controller, combined, output_file):
def save_output(expanded, patterns, controller, combined, output_file):

    expanded_df = pd.DataFrame([
        vars(x) for x in expanded
    ])
    unique_values_df = expanded_df.drop_duplicates(subset=["parsed"])

    pattern_df = pd.DataFrame([
        {
            "Pattern": x.pattern,
            "AbsolutePattern": x.pattern,
            # "ValueType" : x.detailed_type_new,
            "ValueType": remove_after_x(x.detailed_type_new),
            "DetailedValueTypeNew": x.detailed_type_new,
            # "DetailedValueType": x.detailed_type


        }
        for x in patterns
    ])

    controller_df = pd.DataFrame([
        vars(x) for x in controller
    ])

    combined_df = pd.DataFrame(combined)

    # ✅ CREATE F_Combined BEFORE writer
    f_combined_df = combined_df[
        [
            "DataDefinition",
            "AcceptedFeatureName",
            "PartCount",
            # "FeatureCode",
            # "DetailedValueType",
            "ValueID",
            "AcceptedValue",
            "Pattern",
            "AttributeValue",
            "AttributeName",
            "FeatureCode_new",
            "AttributeName_new",
            "DetailedValueTypeNew",
        ]
    ]

    # Optional: remove duplicates
    f_combined_df = f_combined_df.drop_duplicates()

    with pd.ExcelWriter(output_file) as writer:
        expanded_df.to_excel(writer, "String table", index=False)
        pattern_df.to_excel(writer, "Pattern", index=False)
        controller_df.to_excel(writer, "Controller", index=False)
        combined_df.to_excel(writer, "Combined", index=False)
            # ✅ NEW TAB
        f_combined_df.to_excel(writer, "F_Combined", index=False)
        unique_values_df.to_excel(writer, "Unique Values", index=False)   # ← ADD THIS

# from loader import load_data
# from processor import ValueProcessor
# from writer import save_output
# from config import CONFIG


def main_string():

    input_file = "/content/test.xlsx"
    acc_terms_file = "/content/acc-term-Demo.xlsx"

    df, acc_terms_set,acc_terms_set_lower = load_data(
        input_file,
        acc_terms_file,
        CONFIG["accepted_terms_column"]
    )

    processor = ValueProcessor(acc_terms_set, CONFIG)

    all_expanded = []
    all_patterns = []
    all_controller = []
    all_combined = []

    seen_patterns = set()   # ✅ prevent duplicates

    for _, row in df.iterrows():

        if not str(row[CONFIG["value_column"]]).strip():
            continue

        expanded, pattern, controller, combined = processor.process_row(row)

        all_expanded.extend(expanded)
        all_controller.extend(controller)
        all_combined.extend(combined)

        # ✅ unique patterns only
        key = (pattern.pattern, pattern.detailed_type, pattern.detailed_type_new)
        if key not in seen_patterns:
            seen_patterns.add(key)
            all_patterns.append(pattern)

    save_output(
        all_expanded,
        all_patterns,
        all_controller,
        all_combined,
        CONFIG["output_file"]
    )
    print("✅ Done!")

# if __name__ == "__main__":
#     main()

In [2]:
#!/usr/bin/env python3

"""

run_pipeline.py  –  End-to-end driver for the ACC Unified Pipeline

(Colab-friendly version that uses the *new* prefix-aware modules)



Steps:

  1. Ensure mapping.xlsx (local or via GitHub API download)

  2. Pre-process the input (robust CSV/XLSX handling)

  3. Run the *fixed* processing pipeline (prefix-aware wrapper)

  4. Run the *detailed* analysis pipeline (prefix-aware wrapper)

  5. Combine fixed + detailed outputs into final report

"""



#───────────────────────────────────────────────────────────────────────────

# 0  Colab-only pip installs  (harmless elsewhere)

#───────────────────────────────────────────────────────────────────────────

try:

    import google.colab  # noqa: F401

    IN_COLAB = True

except ImportError:

    IN_COLAB = False



if IN_COLAB:

    # Quiet install to avoid extra output in Colab

    # %pip install -q pandas openpyxl streamlit pdfplumber requests pyngrok
    # %pip install -q --ignore-installed pandas openpyxl streamlit pdfplumber requests pyngrok
    %pip install -q --ignore-installed blinker
    %pip install -q "pandas==2.2.2" "requests==2.32.4" "tornado==6.5.1" openpyxl streamlit pdfplumber pyngrok



#───────────────────────────────────────────────────────────────────────────

# 1  Standard libs

#───────────────────────────────────────────────────────────────────────────

import os

import sys

import argparse

import pandas as pd



#───────────────────────────────────────────────────────────────────────────

# 2  Colab upload hook

#───────────────────────────────────────────────────────────────────────────

if IN_COLAB:

    from google.colab import files

    print("🔔  Colab detected – please upload your input file (CSV or XLSX)…")

    uploaded = files.upload()

    if not uploaded:

        sys.exit("No file uploaded; exiting.")

    # Fake argv so argparse sees only the uploaded filename

    sys.argv = [sys.argv[0], next(iter(uploaded.keys()))]



#───────────────────────────────────────────────────────────────────────────

# 3  Optional GitHub token (only needed for API download of mapping.xlsx)

#───────────────────────────────────────────────────────────────────────────

os.environ.setdefault("GITHUB_TOKEN", "")

os.environ.setdefault("GITHUB_OWNER", "Hima9791")

os.environ.setdefault("GITHUB_REPO",  "map")

os.environ.setdefault("GITHUB_FILE_PATH", "mapping.xlsx")



#───────────────────────────────────────────────────────────────────────────

# 4  Project-module imports  (all new wrappers included)

#───────────────────────────────────────────────────────────────────────────

from mapping_utils      import read_mapping_file, save_mapping_to_disk

from github_utils       import download_mapping_file_from_github

from preprocessor       import preprocess_input_file



# NEW prefix-aware wrappers

from pipeline_wrappers  import (

    run_fixed_pipeline_with_prefix_support,

    run_detailed_analysis_with_prefix_support

)



from result_combiner    import combine_results



#───────────────────────────────────────────────────────────────────────────

# 5  Helper: ensure mapping.xlsx exists (local or GitHub)

#───────────────────────────────────────────────────────────────────────────

def ensure_mapping(path: str, allow_github: bool) -> None:

    """Guarantee mapping.xlsx is present at *path* (download if allowed)."""

    if os.path.exists(path):

        print(f"[✔] Found mapping file at '{path}'.")

        return

    if not allow_github:

        sys.exit(f"[✖] mapping file '{path}' not found. "

                 f"Run with --use-github to download it.")

    print("[ ] mapping.xlsx not found – downloading via GitHub API…")

    df_map = download_mapping_file_from_github()

    save_mapping_to_disk(df_map, path)

    print(f"[✔] mapping.xlsx saved to '{path}'.")



#───────────────────────────────────────────────────────────────────────────

# 6  Main driver

#───────────────────────────────────────────────────────────────────────────

def main() -> None:

    # ── CLI args ──────────────────────────────────────────────────

    ap = argparse.ArgumentParser(description="ACC Unified Pipeline runner")

    ap.add_argument("input_excel", help="Input file (CSV or XLSX with 'Value' column)")

    ap.add_argument("--mapping", default="mapping.xlsx",

                    help="Local mapping file name (default: mapping.xlsx)")

    ap.add_argument("--use-github", action="store_true",

                    help="If mapping.xlsx missing, download via GitHub API")

    ap.add_argument("--workdir", default=".",

                    help="Working directory for outputs")

    args, _ = ap.parse_known_args()



    # ── Convert CSV→XLSX if needed ───────────────────────────────

    input_path = args.input_excel

    if input_path.lower().endswith(".csv"):

        print(f"[ ] Detected CSV '{input_path}' – converting to Excel…")

        df_csv = pd.read_csv(input_path)

        input_path = input_path.rsplit(".", 1)[0] + "_converted.xlsx"

        df_csv.to_excel(input_path, index=False, engine="openpyxl")

        print(f"[✔] Saved converted file '{input_path}'")



    # ── Working directory + mapping ───────────────────────────────

    wd = args.workdir.rstrip("/")

    os.makedirs(wd, exist_ok=True)

    mapping_path = os.path.join(wd, args.mapping)

    ensure_mapping(mapping_path, args.use_github)



    #───────────────────────────────────────────────────────────────

    # 1  Pre-processing

    #───────────────────────────────────────────────────────────────

    print("[ ] Pre-processing input…")

    try:

        df_pre = preprocess_input_file(input_path)

    except Exception as e:

        sys.exit(f"[✖] Pre-processing failed: {e}")

    pre_xlsx = os.path.join(wd, "preprocessed.xlsx")

    df_pre.to_excel(pre_xlsx, index=False, engine="openpyxl")

    print(f"[✔] Preprocessed data → {pre_xlsx}")



    #───────────────────────────────────────────────────────────────

    # 2  Fixed pipeline  (prefix-aware)

    #───────────────────────────────────────────────────────────────

    print("[ ] Running fixed pipeline (prefix support)…")

    with open(pre_xlsx, "rb") as fh:

        df_fixed = run_fixed_pipeline_with_prefix_support(fh.read(), mapping_path)

    fixed_xlsx = os.path.join(wd, "fixed_output.xlsx")

    df_fixed.to_excel(fixed_xlsx, index=False, engine="openpyxl")

    print(f"[✔] Fixed output → {fixed_xlsx}")



    #───────────────────────────────────────────────────────────────

    # 3  Detailed analysis  (prefix-aware)

    #───────────────────────────────────────────────────────────────

    print("[ ] Running detailed analysis (prefix support)…")

    detailed_xlsx = os.path.join(wd, "detailed_analysis.xlsx")

    ok = run_detailed_analysis_with_prefix_support(

        input_df=df_pre,

        mapping_file=mapping_path,

        output_file=detailed_xlsx

    )

    if ok is None:

        sys.exit("[✖] Detailed analysis failed.")

    print(f"[✔] Detailed output → {detailed_xlsx}")



    #───────────────────────────────────────────────────────────────

    # 4  Combine fixed + detailed

    #───────────────────────────────────────────────────────────────

    print("[ ] Combining results…")

    final_xlsx = os.path.join(wd, "final_combined.xlsx")

    combined_ok = combine_results(

        processed_df=df_fixed,

        analysis_file=detailed_xlsx,

        output_file=final_xlsx

    )

    if combined_ok is None:

        sys.exit("[✖] Combining results failed.")

    print(f"[✔] Final combined report → {final_xlsx}")



    #───────────────────────────────────────────────────────────────

    # 5  Summary

    #───────────────────────────────────────────────────────────────

    print("\n🎉  Pipeline completed successfully!")

    for label, path in [

        ("Preprocessed",      pre_xlsx),

        ("Fixed pipeline",    fixed_xlsx),

        ("Detailed analysis", detailed_xlsx),

        ("Final report",      final_xlsx)

    ]:

        print(f"   • {label}: {path}")



#───────────────────────────────────────────────────────────────────────────

# 7  Entrypoint

#───────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    main()
    main_string()


🔔  Colab detected – please upload your input file (CSV or XLSX)…


2026-06-17 10:58:09.344 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Saving test_valueSplited_ibrahim.xlsx to test_valueSplited_ibrahim (1).xlsx
[✔] Found mapping file at './mapping.xlsx'.
[ ] Pre-processing input…
[✔] Preprocessed data → ./preprocessed.xlsx
[ ] Running fixed pipeline (prefix support)…


2026-06-17 10:58:09.620 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-06-17 10:58:09.622 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:09.625 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:10.277 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:10.278 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:10.280 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:10.296 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:10.296 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running 

[✔] Fixed output → ./fixed_output.xlsx
[ ] Running detailed analysis (prefix support)…


2026-06-17 10:58:35.271 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:35.272 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:35.275 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:35.484 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:35.486 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:35.486 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:35.549 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:35.550 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

[✔] Detailed output → ./detailed_analysis.xlsx
[ ] Combining results…


2026-06-17 10:58:35.846 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:35.847 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:35.848 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:35.849 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:35.849 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:35.850 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:35.851 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 10:58:35.852 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

[✔] Final combined report → ./final_combined.xlsx

🎉  Pipeline completed successfully!
   • Preprocessed: ./preprocessed.xlsx
   • Fixed pipeline: ./fixed_output.xlsx
   • Detailed analysis: ./detailed_analysis.xlsx
   • Final report: ./final_combined.xlsx
